In [1]:
import json

def clean_key(key):
    # Skip all o: keys except o:label → rdfs:label
    if key == "o:label":
        return "rdfs:label"
        
    if key.startswith("o:"):
        return None

    # wd:Pxxx → wdt:Pxxx
    if key.startswith("wd:P"):
        return "wdt:" + key[3:]

    # wdt:Qxxx → wd:Qxxx
    if key.startswith("wdt:Q"):
        return "wd:" + key[5:]

    return key

def transform(obj):
    if isinstance(obj, dict):
        new_obj = {}
        for k, v in obj.items():
            # Remove unwanted metadata
            if k in ["property_id", "is_public", "thumbnail_display_urls"]:
                continue

            # Convert keys
            new_key = clean_key(k)
            if new_key is None:
                continue

            # Special @id replacement
            if new_key == "@id" and isinstance(v, str) and "/api/items/" in v:
                item_id = v.split("/")[-1]
                new_obj["@id"] = f"http://wendang-project.github.io/data/item/{item_id}"
            # Remove o:Item from @type arrays
            elif new_key == "@type" and isinstance(v, list):
                new_obj["@type"] = [i for i in v if i != "o:Item"]
            else:
                new_obj[new_key] = transform(v)

        if "@language" in new_obj:
            if new_obj["@language"] == "pny":
                new_obj["@language"] = "zh-Latn"

        return new_obj

    elif isinstance(obj, list):
        return [transform(i) for i in obj]
    else:
        return obj

# Load the file
with open("items_omeka.jsonld", "r", encoding="utf-8") as f:
    data = json.load(f)

# Transform the structure
cleaned_data = transform(data)

# Define the @context dictionary
context = {
    "wd": "http://www.wikidata.org/entity/",
    "wdt": "http://www.wikidata.org/prop/direct/",
    "write": "https://w3id.org/write/ontology#",
    "rdfs": "http://www.w3.org/2000/01/rdf-schema#",
    "xsd": "http://www.w3.org/2001/XMLSchema#", 
    "skos": "http://www.w3.org/2004/02/skos/core#"
}

# If the result is a list, add @context to each item
if isinstance(cleaned_data, list):
    for item in cleaned_data:
        if isinstance(item, dict):
            item["@context"] = context
else:
    # Single object case
    if isinstance(cleaned_data, dict):
        cleaned_data["@context"] = context

# Save output
with open("data1.jsonld", "w", encoding="utf-8") as f:
    json.dump(cleaned_data, f, indent=2, ensure_ascii=False)

print("✅ Cleaning complete")


✅ Cleaning complete
